In [18]:
## Import modules

import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

In [19]:
url = f"https://www.framesdirect.com/eyeglasses/"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://google.com"
}

response = requests.get(url, headers=headers)

print(f"Status code: {response.status_code}")

Status code: 200


In [40]:
brand_name = []
product_name = []
product_price = []
color = []
rating = []

In [41]:
soup = BeautifulSoup(response.content, 'html.parser')

In [42]:
container = soup.find('div', id='product-list-container')

In [30]:
container

<div class="d-flex flex-wrap align-content-around justify-content-start px-2" data-nosnippet="" id="product-list-container">
<div class="prod-holder" data-description="Oakley_Airdrop_888392593948" data-element-id="Tiles_Tile0"><a aria-label="#ProductAriaLabel#" class="prod-tile-wrapper decoration-none" href="/oakley-airdrop-eyeglasses?color=7" id="aTiles_Tile0" tabindex="-1"></a><div class="abs-container"><div class="abs-icon-speedy-rx"></div><div class="abs-icon-sale-container"><div class="abs-icon-sale-badge"></div></div><div class="abs-icon-popular"></div></div><div class="prod-image-holder"><a data-objectid="10005944-7-10833445" data-position="1" data-queryid="2e08792446b9ea3bba241f04454b9525" href="/oakley-airdrop-eyeglasses?color=7" id="aProdutImageLink09610005944"><div class="imageholder glasses-wrapper-hover-singleImg"><img alt="Visit Eyeglasses Oakley Airdrop Image 1" fetchpriority="high" height="750" id="imgSrc09610005944" src="https://media.framesdirect.com/product_elarge_im

In [44]:
all_glasses = container.find_all('div', class_='prod-holder') if container else []

In [32]:
all_glasses

[<div class="prod-holder" data-description="Oakley_Airdrop_888392593948" data-element-id="Tiles_Tile0"><a aria-label="#ProductAriaLabel#" class="prod-tile-wrapper decoration-none" href="/oakley-airdrop-eyeglasses?color=7" id="aTiles_Tile0" tabindex="-1"></a><div class="abs-container"><div class="abs-icon-speedy-rx"></div><div class="abs-icon-sale-container"><div class="abs-icon-sale-badge"></div></div><div class="abs-icon-popular"></div></div><div class="prod-image-holder"><a data-objectid="10005944-7-10833445" data-position="1" data-queryid="2e08792446b9ea3bba241f04454b9525" href="/oakley-airdrop-eyeglasses?color=7" id="aProdutImageLink09610005944"><div class="imageholder glasses-wrapper-hover-singleImg"><img alt="Visit Eyeglasses Oakley Airdrop Image 1" fetchpriority="high" height="750" id="imgSrc09610005944" src="https://media.framesdirect.com/product_elarge_images/Oakley-Airdropa-Grey-Shadow-804603-Eyeglasses.jpg?impolicy=fdresize&amp;wid=316" title="Visit Eyeglasses Oakley Airdrop

In [33]:
for glass in all_glasses:
    try:
        brand = glass.find('div', class_='catalog-name').text.strip()
        brand_name.append(brand)
        
        product = glass.find('div', class_='product-name').text.strip()
        product_name.append(product)
        
        price = glass.find('div', class_='prod-aslowas').text.strip()
        product_price.append(price)
        
        color_span = glass.find('span', id=re.compile(r'^dvSelectedColorName'))
        glass_color = color_span.text.strip() if color_span else "Unknwon"
        color.append(glass_color)
        
        popular_block = glass.find('div', class_='prod-popular')
        if popular_block:
            star_count = len(popular_block.find_all('i', class_='icon-star-solid'))
            rating_value = f"{star_count} Stars" if star_count > 0 else "No reviews"
        else:
            rating_value = "No Review"
        rating.append(rating_value)
    except AttributeError:
        pass

    

print(f"Extraction complete! Found {len(product_name)} items.")

Extraction complete! Found 0 items.


In [45]:
for glass in all_glasses:
    try:
        # 1. Extract Brand safely (Dropped 'div' tag limit)
        brand_el = glass.find(class_='catalog-name')
        brand = brand_el.text.strip() if brand_el else "Unknown Brand"
        
        # 2. Extract Product Name safely (Accounts for product-name or product_name)
        product_el = glass.find(class_='product-name') or glass.find(class_='product_name')
        product = product_el.text.strip() if product_el else "Unknown Product"
        
        # 3. Extract Product Price safely 
        price_el = glass.find(class_='prod-aslowas')
        price = price_el.text.strip() if price_el else "Unknown Price"
        
        # 4. Extract Color Name safely
        color_span = glass.find('span', id=re.compile(r'^dvSelectedColorName'))
        glass_color = color_span.text.strip() if color_span else "Unknown Color"
        
        # 5. Extract Star Rating safely
        popular_block = glass.find(class_='prod-popular')
        if popular_block:
            star_count = len(popular_block.find_all('i', class_='icon-star-solid'))
            rating_value = f"{star_count} Stars" if star_count > 0 else "No Reviews"
        else:
            rating_value = "No Reviews"
            
        # APPEND AT THE VERY END ONLY AFTER VARIABLES PASS SAFELY
        brand_name.append(brand)
        product_name.append(product)
        product_price.append(price)
        color.append(glass_color)
        rating.append(rating_value)
        
    except Exception as e:
        # Changed 'pass' to a print flag so you can explicitly see if any background errors slip through
        print(f"A card encountered an unexpected error: {e}")

print(f"Extraction complete! Found {len(product_name)} items.")

Extraction complete! Found 25 items.


In [46]:
print(len(brand_name), len(product_name), len(product_price), len(color), len(rating))

25 25 25 25 25


In [47]:
glass_df = pd.DataFrame({
    "Brand": brand_name,
    "Product Name": product_name,
    "Price": product_price,
    "Color": color,
    "Rating": rating
})


In [48]:
glass_df

,Brand,Product Name,Price,Color,Rating
0,Oakley,Airdrop,$227,Satin Black (OX8046-01),5 Stars
1,Ray-Ban,RB7047,$176,Matte Black (5196),5 Stars
2,Unknown Brand,Unknown Product,Unknown Price,Unknown Color,No Reviews
3,RALPH by Ralph Lauren,RA7158U,$105.75,Shiny Transparent Beige (6117),No Reviews
4,Oakley,Plank 2.0,$187,Polished Grey Smoke (OX8081-06),5 Stars
5,Armani Exchange,AX1017,$103.50,Black (6000),5 Stars
6,Polo,PH2126,$145.50,Matte Black-Grey (5534),5 Stars
7,Gucci,GG1424O,$365.50,Black (005),No Reviews
8,Ray-Ban,RB6355,$237,Matte Black (2503),5 Stars
9,Oakley,Sway Bar 0.5,$387,Pewter (508002),No Reviews
